# Ejercicio Guiado Introductorio de Inteligencia Artificial

## Tema integrador
Construccion de un agente repartidor que debe encontrar rutas en un campus y tomar decisiones apoyadas por aprendizaje supervisado.

## Proposito del notebook
Este notebook esta pensado como una primera practica guiada. La idea no es solo ejecutar codigo, sino entender por que cada parte representa un concepto de Inteligencia Artificial visto en las infografias.

## Competencias que se trabajan
- Identificar que es un agente racional.
- Describir un problema con el esquema PEAS.
- Representar un entorno como una cuadricula o espacio de estados.
- Implementar BFS y DFS paso a paso.
- Comparar el comportamiento de ambos algoritmos.
- Comprender como el aprendizaje supervisado complementa a un agente.

## Recomendacion de uso
Ejecuta las celdas en orden. Lee cada comentario del codigo y responde las preguntas en las secciones de reflexion.


## Parte 1. Activacion de conocimientos previos
Antes de programar, piensa en estas preguntas:

1. Que hace que un sistema pueda considerarse IA y no solo un programa tradicional?
2. Cual es la diferencia entre buscar una solucion y aprender una decision?
3. El agente de este ejercicio pertenece a ANI, AGI o ASI?
4. Que limitaciones podria tener un agente como este en la vida real?


In [1]:
# Librerias basicas que usaremos en el notebook.
# Solo usamos herramientas de Python puro para que el ejercicio sea facil de ejecutar.
from collections import deque
import math
import random

# Fijamos la semilla para que los datos aleatorios sean reproducibles.
random.seed(42)


## Parte 2. Modelado del problema como agente racional
En IA, un agente racional es aquel que selecciona acciones buscando el mejor resultado posible segun una medida de rendimiento.

Para aterrizar este concepto vamos a usar PEAS:
- **P**: Performance measure
- **E**: Environment
- **A**: Actuators
- **S**: Sensors


In [2]:
# Definimos una primera propuesta de PEAS para el agente repartidor.
# Puedes modificarla si deseas adaptar el caso a otro contexto.
peas = {
    "Performance": [
        "entregar el paquete en la meta",
        "usar la menor cantidad de pasos posible",
        "evitar obstaculos o zonas no transitables"
    ],
    "Environment": "campus representado como una cuadricula con celdas libres y obstaculos",
    "Actuators": ["arriba", "abajo", "izquierda", "derecha"],
    "Sensors": ["posicion actual", "ubicacion de la meta", "celdas vecinas disponibles"]
}

peas


{'Performance': ['entregar el paquete en la meta',
  'usar la menor cantidad de pasos posible',
  'evitar obstaculos o zonas no transitables'],
 'Environment': 'campus representado como una cuadricula con celdas libres y obstaculos',
 'Actuators': ['arriba', 'abajo', 'izquierda', 'derecha'],
 'Sensors': ['posicion actual',
  'ubicacion de la meta',
  'celdas vecinas disponibles']}

### Reflexion guiada
Completa la clasificacion del entorno.

Sugerencia: piensa si el agente ve todo el tablero, si las acciones cambian siempre de la misma forma el estado, si cada decision depende de la anterior y si el entorno cambia mientras el agente actua.


In [3]:
clasificacion_entorno = {
    "observable": "Completar",
    "determinista": "Completar",
    "episodico_o_secuencial": "Completar",
    "estatico_o_dinamico": "Completar"
}

clasificacion_entorno


{'observable': 'Completar',
 'determinista': 'Completar',
 'episodico_o_secuencial': 'Completar',
 'estatico_o_dinamico': 'Completar'}

## Parte 3. Representacion del entorno
Una forma muy comun de representar problemas de busqueda es convertir el entorno en un espacio de estados.

En este caso:
- cada celda de la cuadricula sera un posible estado,
- el agente podra moverse a celdas vecinas,
- algunas celdas seran obstaculos,
- el objetivo sera pasar desde `S` hasta `G`.


In [4]:
# S = Start o inicio
# G = Goal o meta
# . = espacio libre
# # = obstaculo
grid = [
    ["S", ".", ".", "#", ".", "."],
    [".", "#", ".", "#", ".", "."],
    [".", "#", ".", ".", ".", "#"],
    [".", ".", "#", "#", ".", "."],
    ["#", ".", ".", ".", "#", "."],
    [".", ".", "#", ".", ".", "G"]
]

# Mostramos el tablero de forma amigable.
for row in grid:
    print(" ".join(row))


S . . # . .
. # . # . .
. # . . . #
. . # # . .
# . . . # .
. . # . . G


In [5]:
def find_symbol(grid, symbol):
    """Busca un simbolo dentro de la cuadricula y devuelve su posicion."""
    for i, row in enumerate(grid):
        for j, value in enumerate(row):
            if value == symbol:
                return (i, j)
    return None

def valid_neighbors(grid, node):
    """Devuelve los vecinos validos de una celda.
    Un vecino es valido si esta dentro del tablero y no es obstaculo.
    """
    rows = len(grid)
    cols = len(grid[0])
    i, j = node

    # Movimientos permitidos: arriba, abajo, izquierda y derecha.
    candidates = [(i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)]
    neighbors = []

    for x, y in candidates:
        if 0 <= x < rows and 0 <= y < cols and grid[x][y] != "#":
            neighbors.append((x, y))

    return neighbors

start = find_symbol(grid, "S")
goal = find_symbol(grid, "G")

print("Inicio:", start)
print("Meta:", goal)
print("Vecinos del inicio:", valid_neighbors(grid, start))


Inicio: (0, 0)
Meta: (5, 5)
Vecinos del inicio: [(1, 0), (0, 1)]


## Parte 4. Busqueda no informada
Ahora implementaremos dos algoritmos clasicos:

- **BFS** explora por niveles. En problemas con costos uniformes, encuentra la ruta mas corta.
- **DFS** profundiza primero. Puede encontrar una solucion rapido, pero no siempre la mejor.

La meta de esta parte es que observes no solo el resultado, sino tambien el orden en que los nodos son explorados.


In [6]:
def reconstruct_path(parents, goal):
    """Reconstruye el camino desde la meta hasta el inicio usando el diccionario de padres."""
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parents[current]

    # Invertimos el camino porque lo reconstruimos desde la meta hacia el inicio.
    return list(reversed(path))

def bfs(grid, start, goal):
    """Busqueda en anchura. Usa una cola FIFO."""

    # La cola empieza con el nodo inicial.
    queue = deque([start])

    # Guardamos desde que nodo llegamos a cada estado.
    parents = {start: None}

    # Guardamos el orden de visita para analizar el comportamiento del algoritmo.
    visited_order = []

    while queue:
        current = queue.popleft()
        visited_order.append(current)

        # Si llegamos a la meta, reconstruimos el camino y terminamos.
        if current == goal:
            return reconstruct_path(parents, goal), visited_order

        # Recorremos vecinos del nodo actual.
        for neighbor in valid_neighbors(grid, current):
            # Si no lo hemos descubierto antes, lo agregamos a la cola.
            if neighbor not in parents:
                parents[neighbor] = current
                queue.append(neighbor)

    return None, visited_order

def dfs(grid, start, goal):
    """Busqueda en profundidad. Usa una pila LIFO."""

    stack = [start]
    parents = {start: None}
    visited_order = []

    while stack:
        current = stack.pop()
        visited_order.append(current)

        if current == goal:
            return reconstruct_path(parents, goal), visited_order

        # Usamos reversed para conservar un orden de exploracion mas claro.
        for neighbor in reversed(valid_neighbors(grid, current)):
            if neighbor not in parents:
                parents[neighbor] = current
                stack.append(neighbor)

    return None, visited_order


In [7]:
# Ejecutamos ambos algoritmos sobre el mismo entorno.
path_bfs, visited_bfs = bfs(grid, start, goal)
path_dfs, visited_dfs = dfs(grid, start, goal)

print("Resultado de BFS")
print("Ruta encontrada:", path_bfs)
print("Cantidad de pasos:", len(path_bfs) - 1)
print("Orden de visita:", visited_bfs)
print()
print("Resultado de DFS")
print("Ruta encontrada:", path_dfs)
print("Cantidad de pasos:", len(path_dfs) - 1)
print("Orden de visita:", visited_dfs)


Resultado de BFS
Ruta encontrada: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (4, 2), (4, 3), (5, 3), (5, 4), (5, 5)]
Cantidad de pasos: 10
Orden de visita: [(0, 0), (1, 0), (0, 1), (2, 0), (0, 2), (3, 0), (1, 2), (3, 1), (2, 2), (4, 1), (2, 3), (5, 1), (4, 2), (2, 4), (5, 0), (4, 3), (1, 4), (3, 4), (5, 3), (0, 4), (1, 5), (3, 5), (5, 4), (0, 5), (4, 5), (5, 5)]

Resultado de DFS
Ruta encontrada: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (4, 2), (4, 3), (5, 3), (5, 4), (5, 5)]
Cantidad de pasos: 10
Orden de visita: [(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (4, 1), (5, 1), (5, 0), (4, 2), (4, 3), (5, 3), (5, 4), (5, 5)]


### Analisis guiado
Responde en tus propias palabras:

1. Cual algoritmo exploro por niveles?
2. Cual algoritmo garantiza la ruta mas corta cuando todos los pasos cuestan lo mismo?
3. Que ventaja y que desventaja observas en DFS?
4. Por que este problema pertenece al tema de busqueda no informada?


In [8]:
# Actividad practica sugerida:
# 1. Cambia algunos obstaculos del tablero.
# 2. Vuelve a ejecutar esta celda.
# 3. Observa si cambian la ruta y el orden de visita.
grid_modificado = [row[:] for row in grid]

# Ejemplo de cambio guiado: liberamos una celda y bloqueamos otra.
grid_modificado[0][3] = "."
grid_modificado[4][3] = "#"

for row in grid_modificado:
    print(" ".join(row))

nuevo_inicio = find_symbol(grid_modificado, "S")
nueva_meta = find_symbol(grid_modificado, "G")
nuevo_bfs, _ = bfs(grid_modificado, nuevo_inicio, nueva_meta)
nuevo_dfs, _ = dfs(grid_modificado, nuevo_inicio, nueva_meta)

print()
print("Nueva ruta BFS:", nuevo_bfs)
print("Nueva ruta DFS:", nuevo_dfs)


S . . . . .
. # . # . .
. # . . . #
. . # # . .
# . . # # .
. . # . . G

Nueva ruta BFS: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (3, 5), (4, 5), (5, 5)]
Nueva ruta DFS: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (3, 5), (4, 5), (5, 5)]


## Parte 5. Puente hacia busqueda informada
Aunque aqui no implementaremos un algoritmo completo de busqueda informada, si construiremos una heuristica simple.

La distancia Manhattan estima cuantos movimientos horizontales y verticales hacen falta para llegar a la meta.


In [9]:
def manhattan(a, b):
    """Calcula la distancia Manhattan entre dos posiciones."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

sample_nodes = [start, (2, 2), (4, 3), goal]
for node in sample_nodes:
    print(f"Nodo {node} -> heuristica = {manhattan(node, goal)}")


Nodo (0, 0) -> heuristica = 10
Nodo (2, 2) -> heuristica = 6
Nodo (4, 3) -> heuristica = 3
Nodo (5, 5) -> heuristica = 0


### Reflexion
Si un algoritmo conociera esta heuristica, como podria usarla para priorizar nodos mas prometedores?


## Parte 6. Aprendizaje supervisado introductorio
Hasta ahora el agente ha buscado rutas. Ahora agregaremos una segunda capacidad: aprender a decidir si una ruta parece conveniente o no, usando ejemplos historicos.

Esto no reemplaza a BFS o DFS. Mas bien complementa al agente con una prediccion basada en datos.


In [ ]:
# Generamos datos simulados de rutas anteriores.
# Cada fila representa una situacion observada por el agente.
rows = []

for _ in range(120):
    distancia = random.randint(4, 20)
    congestion = random.randint(1, 10)
    obstaculos = random.randint(0, 6)
    clima = random.randint(0, 1)

    # Esta regla crea la etiqueta objetivo.
    # Mientras menor sea el score, mas conveniente consideramos la ruta.
    score = distancia + congestion + obstaculos * 1.5 + clima * 3
    conveniente = 1 if score <= 20 else 0

    rows.append({
        "distancia": distancia,
        "congestion": congestion,
        "obstaculos": obstaculos,
        "clima": clima,
        "conveniente": conveniente
    })

# Mostramos solo las primeras observaciones.
rows[:5]


Trabajaremos con dos modelos sencillos:

- **KNN**: compara un caso nuevo con ejemplos parecidos.
- **Decision Stump**: es una regla simple basada en una sola variable y un umbral.

No usaremos librerias externas para que el codigo sea facil de leer.


In [ ]:
feature_names = ["distancia", "congestion", "obstaculos", "clima"]

def train_test_split_simple(data, test_ratio=0.25):
    """Divide los datos en entrenamiento y prueba."""
    data_copy = data[:]
    random.shuffle(data_copy)
    cut = int(len(data_copy) * (1 - test_ratio))
    return data_copy[:cut], data_copy[cut:]

def vectorize(example):
    """Convierte un ejemplo en una lista numerica de caracteristicas."""
    return [example[name] for name in feature_names]

def euclidean_distance(a, b):
    """Calcula la distancia euclidiana entre dos vectores."""
    return math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))

def knn_predict(train_data, example, k=5):
    """Predice la clase de un ejemplo segun sus k vecinos mas cercanos."""
    distances = []
    target_vector = vectorize(example)

    for row in train_data:
        dist = euclidean_distance(vectorize(row), target_vector)
        distances.append((dist, row["conveniente"]))

    # Ordenamos de menor a mayor distancia.
    distances.sort(key=lambda item: item[0])
    nearest = distances[:k]

    # Sumamos votos de la clase 1.
    votes = sum(label for _, label in nearest)
    return 1 if votes >= (k / 2) else 0

def learn_decision_stump(train_data):
    """Aprende una regla simple basada en una sola variable y un umbral."""
    best_feature = None
    best_threshold = None
    best_accuracy = -1

    for feature in feature_names:
        values = sorted({row[feature] for row in train_data})

        for threshold in values:
            predictions = [1 if row[feature] <= threshold else 0 for row in train_data]
            real = [row["conveniente"] for row in train_data]
            accuracy = sum(int(p == y) for p, y in zip(predictions, real)) / len(real)

            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_feature = feature
                best_threshold = threshold

    return {
        "feature": best_feature,
        "threshold": best_threshold,
        "train_accuracy": best_accuracy
    }

def stump_predict(model, example):
    """Aplica la regla aprendida a un ejemplo nuevo."""
    return 1 if example[model["feature"]] <= model["threshold"] else 0

def accuracy_score_simple(real, pred):
    """Calcula accuracy de forma manual."""
    return sum(int(a == b) for a, b in zip(real, pred)) / len(real)

def confusion_matrix_simple(real, pred):
    """Construye una matriz de confusion basica."""
    tp = sum(1 for a, b in zip(real, pred) if a == 1 and b == 1)
    tn = sum(1 for a, b in zip(real, pred) if a == 0 and b == 0)
    fp = sum(1 for a, b in zip(real, pred) if a == 0 and b == 1)
    fn = sum(1 for a, b in zip(real, pred) if a == 1 and b == 0)
    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}


In [ ]:
# Dividimos los datos.
train_data, test_data = train_test_split_simple(rows, test_ratio=0.25)
real_test = [row["conveniente"] for row in test_data]

# Predicciones con KNN.
pred_knn = [knn_predict(train_data, row, k=5) for row in test_data]

# Entrenamiento y prediccion con una regla simple.
stump_model = learn_decision_stump(train_data)
pred_stump = [stump_predict(stump_model, row) for row in test_data]

print("Accuracy KNN:", round(accuracy_score_simple(real_test, pred_knn), 3))
print("Accuracy Stump:", round(accuracy_score_simple(real_test, pred_stump), 3))
print("Modelo aprendido por Stump:", stump_model)


In [ ]:
print("Matriz de confusion KNN")
print(confusion_matrix_simple(real_test, pred_knn))
print()
print("Matriz de confusion Stump")
print(confusion_matrix_simple(real_test, pred_stump))


### Interpretacion guiada
1. Que modelo obtuvo mejor accuracy?
2. Que significa que una ruta sea `conveniente` en este ejemplo?
3. La regla `stump` usa una sola variable. Que ventaja didactica tiene eso?
4. Por que este modelo no reemplaza por completo al algoritmo de busqueda?


In [ ]:
# Probamos ambos modelos con una ruta nueva.
nueva_ruta = {"distancia": 8, "congestion": 3, "obstaculos": 1, "clima": 0}

prediccion_knn = knn_predict(train_data, nueva_ruta, k=5)
prediccion_stump = stump_predict(stump_model, nueva_ruta)

print("Nueva ruta:", nueva_ruta)
print("Prediccion KNN:", "conveniente" if prediccion_knn == 1 else "no conveniente")
print("Prediccion Stump:", "conveniente" if prediccion_stump == 1 else "no conveniente")


## Parte 7. Cierre del ejercicio
Completa estas ideas con tus propias palabras:

- El agente puede considerarse racional porque...
- BFS fue util en este problema porque...
- DFS fue util en este problema porque...
- El aprendizaje supervisado aporto valor porque...
- Una limitacion importante del sistema es...

## Reto opcional
1. Cambia el tablero y diseña un caso donde DFS encuentre una ruta peor que BFS.
2. Crea una nueva variable para el conjunto de datos, por ejemplo `hora_pico`.
3. Explica en que parte del sistema usarias busqueda y en que parte usarias aprendizaje.


# PARTE 7

## Reflexión final

### El agente puede considerarse racional porque...
```
El agente se puede considerar racional porque toma decisiones basadas en un objetivo claro y una medida de rendimiento (PEAS). No actúa al azar, sino que analiza el entorno, evita obstáculos y elige movimientos que lo acerquen a la meta.

En este caso, parte de un estado inicial (S) y busca llegar a una meta (G), tomando decisiones paso a paso de forma lógica. Básicamente, intenta hacer lo “mejor posible” con la información que tiene, usando búsqueda para encontrar una solución.
```

### BFS fue útil en este problema porque...
```
BFS fue útil porque garantiza encontrar la ruta más corta cuando todos los movimientos tienen el mismo costo. Explora el espacio por niveles, como si fuera expandiéndose en capas, lo que asegura que cuando llega a la meta, lo hace por el camino más corto.

Personalmente, me pareció más claro de entender porque es más ordenado y evita perderse en caminos largos innecesarios.
```

### DFS fue útil en este problema porque...
```
DFS es útil porque consume menos memoria y puede encontrar una solución rápido si la meta está en profundidad. En lugar de explorar todo por niveles, se mete directamente por un camino hasta el fondo.

Sin embargo, en este problema noté que puede no ser la mejor opción, ya que no garantiza encontrar el camino más corto y puede terminar recorriendo rutas más largas dependiendo del orden en que explore.
```

### El aprendizaje supervisado aportó valor porque...
```
El aprendizaje supervisado aporta valor porque no solo busca rutas, sino que también ayuda a evaluar qué tan buenas son. Mientras BFS o DFS encuentran caminos posibles, el modelo puede predecir si una ruta es conveniente usando datos como distancia, congestión o clima.

Esto me parece importante porque en la vida real no siempre queremos cualquier ruta, sino la mejor según diferentes factores. Es una forma de combinar lógica con experiencia previa.
```

### Una limitación importante del sistema es...
```
Una limitación importante es que la búsqueda no considera factores dinámicos del mundo real, como cambios en el tráfico o nuevos obstáculos. Además, el modelo de aprendizaje depende de los datos con los que fue entrenado, así que si esos datos no son buenos, las predicciones tampoco lo serán.

También, algoritmos como BFS y DFS no son prácticos en espacios muy grandes, como una ciudad completa, donde se necesitarían métodos más avanzados con heurísticas.
```

---

## RETO 1

```python
# RETO 1: Tablero donde DFS es peor que BFS

grid_reto1 = [
    ["S", ".", ".", ".", ".", "."],
    ["#", "#", "#", "#", "#", "."],
    [".", ".", ".", ".", ".", "."],
    ["#", "#", "#", "#", "#", "."],
    [".", ".", ".", ".", ".", "."],
    ["#", "#", "#", "#", "#", "G"]
]

print("Tablero (forma de serpiente):")
for row in grid_reto1:
    print(" ".join(row))

start_r1 = find_symbol(grid_reto1, "S")
goal_r1 = find_symbol(grid_reto1, "G")

path_bfs_r1, visited_bfs_r1 = bfs(grid_reto1, start_r1, goal_r1)
path_dfs_r1, visited_dfs_r1 = dfs(grid_reto1, start_r1, goal_r1)

print("\nBFS:")
print(f"  Pasos: {len(path_bfs_r1) - 1}")
print(f"  Nodos explorados: {len(visited_bfs_r1)}")

print("\nDFS:")
print(f"  Pasos: {len(path_dfs_r1) - 1}")
print(f"  Nodos explorados: {len(visited_dfs_r1)}")

print(f"\nDiferencia: DFS usó {len(path_dfs_r1) - len(path_bfs_r1)} pasos más")
print("\nExplicación: DFS explora profundidad primero, siguiendo hacia abajo.")
print("Esto lo lleva a recorrer toda la serpiente. BFS encuentra rápidamente")
print("el camino directo a través de los corredores.")
```

---

## RETO 2

```python
# RETO 2: Agregar variable 'hora_pico'

rows_reto2 = []

for _ in range(120):
    distancia = random.randint(4, 20)
    congestion = random.randint(1, 10)
    obstaculos = random.randint(0, 6)
    clima = random.randint(0, 1)
    
    # NUEVA VARIABLE
    hora_pico = random.randint(0, 1)
    
    # Regla actualizada: hora_pico penaliza con 5 puntos
    score = distancia + congestion + obstaculos * 1.5 + clima * 3 + hora_pico * 5
    conveniente = 1 if score <= 20 else 0
    
    rows_reto2.append({
        "distancia": distancia,
        "congestion": congestion,
        "obstaculos": obstaculos,
        "clima": clima,
        "hora_pico": hora_pico,  # Nueva característica
        "conveniente": conveniente
    })

# Análisis
rutas_pico = [r for r in rows_reto2 if r["hora_pico"] == 1]
rutas_no_pico = [r for r in rows_reto2 if r["hora_pico"] == 0]

conv_pico = sum(1 for r in rutas_pico if r["conveniente"] == 1)
conv_no_pico = sum(1 for r in rutas_no_pico if r["conveniente"] == 1)

print("ANÁLISIS DE HORA_PICO:")
print(f"Rutas en hora pico: {len(rutas_pico)}, Convenientes: {conv_pico} "
      f"({100*conv_pico/len(rutas_pico):.1f}%)")
print(f"Rutas sin hora pico: {len(rutas_no_pico)}, Convenientes: {conv_no_pico} "
      f"({100*conv_no_pico/len(rutas_no_pico):.1f}%)")

print("\nConclusión: La hora pico reduce significativamente la probabilidad")
print("de que una ruta sea conveniente, reflejando la realidad urbana.")
```

---

## RETO 3 - Explicación estructurada

### BÚSQUEDA (BFS/DFS) - PARA ENCONTRAR

**Dónde:** Fase de PLANIFICACIÓN
- Encontrar rutas válidas
- Garantizar llegada a meta
- Minimizar distancia

**Cuándo:** Necesitamos solución GARANTIZADA
- Meta está bien definida
- Entorno es estático
- Optimizar costo es crítico

**Ejemplo:** "Encuentra la ruta más corta del edificio A al B"

### APRENDIZAJE (KNN/Stump) - PARA EVALUAR

**Dónde:** Fase de EVALUACIÓN y PRIORIZACIÓN
- Predecir si ruta es conveniente
- Aprender de patrones históricos
- Elegir mejor entre opciones válidas

**Cuándo:** Usamos EXPERIENCIA HISTÓRICA
- Entorno tiene dinámicas (congestión, clima)
- Múltiples rutas válidas
- Tenemos datos históricos

**Ejemplo:** "De 3 rutas válidas, ¿cuál fue mejor en hora pico?"

### ARQUITECTURA COMPLETA

```
Solicitud → [BÚSQUEDA] BFS encuentra rutas válidas
          → [EXTRAE] Características de cada ruta
          → [APRENDIZAJE] Predice conveniencia
          → [ELIGE] La mejor evaluada
          → [EJECUTA] Sigue la ruta
          → [RECOPILA] Datos para mejorar
          → [REENTRENAMIENTO] Periódico del modelo
```

### VENTAJAS DE COMBINAR

- Búsqueda = **FACTIBILIDAD** (existe la ruta)
- Aprendizaje = **UTILIDAD** (es buena según patrones)
- Sistema = **ROBUSTO y ADAPTATIVO**

### PROBLEMA DE SOLO UNO

❌ **Solo búsqueda:** Siempre elige más corta, ignora datos históricos
❌ **Solo aprendizaje:** Predice ruta conveniente que ya no existe

✅ **Ambas:** Garantiza solución válida Y buena según experiencia

---

## Resumen de conceptos clave

| Concepto | Qué es | Para qué | Limitación |
|----------|--------|---------|-----------|
| **Agente Racional** | Elige acciones por PEAS | Optimizar rendimiento | Depende de buena definición de PEAS |
| **BFS** | Explora por niveles | Ruta más corta | Alto uso de memoria |
| **DFS** | Explora por profundidad | Rápido con espacio pequeño | No garantiza óptimo |
| **KNN** | Compara con similares | Predecir categoría | Lento si hay muchos datos |
| **Decision Stump** | Una regla simple | Interpretable | Baja precisión si problema es complejo |
| **Combinación** | Búsqueda + Aprendizaje | Robusto y adaptativo | Necesita ambas para máxima utilidad |